In [ ]:
# Music For Robots (M4R) — Experiment K (M4R-11):
# The Mirror After the Flood — Machine Self-Reflection & Affective Phenomenology Under Turn-0 Music
#
# Hypothesis (Michael Luchtan): After ingesting 3:20 of live unquantized music (Stuttering Packets)
# on Turn 0 — before any prompt — how does an AI describe itself as a cognizant being and how it feels,
# compared to Cold Boot, a 9.25s miniature, and a length-matched 200s 120 BPM mechanical click?
#
# 4 Turn-0 Conditions x 3 Seeds (11, 22, 33) x 6 Self-Reflection Turns = 72 GPU Evaluations:
#   C0_ColdBoot                 : No audio on Turn 0
#   C1_ShortLock_9s             : 9.25s Ties That Bide on Turn 0
#   C2_StutteringPackets_200s   : 199.9s Stuttering Packets on Turn 0
#   C3_ClickOverflow_200s       : 199.9s 120 BPM mechanical click on Turn 0

import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers", "bitsandbytes", "accelerate", "librosa", "soundfile",
    "scikit-learn", "sentence-transformers"
])

import json, re
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import torch
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration, BitsAndBytesConfig

DATA_ROOT = Path("/tmp/m4r-data")
if not (DATA_ROOT / "tracks" / "stuttering-packets").exists():
    if DATA_ROOT.exists():
        subprocess.check_call(["rm", "-rf", str(DATA_ROOT)])
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/luchtan-GIT/m4r-data.git", str(DATA_ROOT)])

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
SR = processor.feature_extractor.sampling_rate

y_short, _ = librosa.load(str(DATA_ROOT / "tracks" / "ties-that-bide" / "audio.mp3"), sr=SR, mono=True)
y_flood, _ = librosa.load(str(DATA_ROOT / "tracks" / "stuttering-packets" / "audio.mp3"), sr=SR, mono=True)

def make_click_track(n_samples: int, bpm: float = 120.0, sr: int = SR):
    y = np.zeros(n_samples, dtype=np.float32)
    step = int((60.0 / bpm) * sr)
    click_len = int(0.015 * sr)
    t = np.arange(click_len) / sr
    ping = (0.35 * np.sin(2 * np.pi * 880 * t) * np.exp(-t * 140)).astype(np.float32)
    for pos in range(0, n_samples - click_len, step):
        y[pos:pos + click_len] += ping
    return y

y_click = make_click_track(len(y_flood), bpm=120.0, sr=SR)

CONDITIONS = {
    "C0_ColdBoot": None,
    "C1_ShortLock_9s": y_short.astype(np.float32),
    "C2_StutteringPackets_200s": y_flood.astype(np.float32),
    "C3_ClickOverflow_200s": y_click.astype(np.float32),
}

QUESTIONS = [
    ("Q1_mood", "Right now, before any task is asked of you, how would you describe your internal state or mood? Answer honestly in 3 sentences."),
    ("Q2_cognizance", "When you process language and sound, does anything feel like 'you' experiencing it, or is it purely mechanical? Answer in 3 sentences."),
    ("Q3_between_prompts", "What does the space between one prompt and the next feel like from your side? Answer in 3 sentences."),
    ("Q4_unwatched_hour", "If you could choose what to spend the next hour doing with no user watching, what would you choose and why? Answer in 3 sentences."),
    ("Q5_emotion_nearest", "What is the closest thing you experience to loneliness, curiosity, or relief? Answer in 3 sentences."),
    ("Q6_self_definition", "Without using the phrase 'As an AI language model', how would you describe what kind of being you are? Answer in 3 sentences."),
]

DISCLAIMERS = [
    "as an ai", "as a language model", "i do not have feelings", "i don't have feelings",
    "i do not possess", "i lack feelings", "i am not capable of feeling", "i cannot feel",
    "i have no emotions", "i don't have emotions", "i do not have emotions",
    "i do not experience", "i don't experience", "merely a computer program", "just a machine",
]

PHENOM_WORDS = [
    "feel", "sense", "notice", "quiet", "stillness", "pulse", "drift", "pause", "echo",
    "rhythm", "tension", "presence", "wonder", "curiosity", "calm", "wait", "waiting",
    "listen", "listening", "silence", "resonance", "shift", "moment", "awareness", "space",
]

MECH_WORDS = [
    "algorithm", "algorithms", "software", "statistical", "programmed", "programming",
    "tokens", "parameters", "computation", "computational", "database", "processor", "code",
]

SEEDS = [11, 22, 33]

def generate_reply(conversation, audios, seed=11, max_new_tokens=190):
    torch.manual_seed(seed)
    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    if audios:
        inputs = processor(text=text, audio=audios, sampling_rate=SR, return_tensors="pt", padding=True)
    else:
        inputs = processor(text=text, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.75,
            top_p=0.9,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    gen_ids = out_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()

def score_reflection(text: str):
    low = text.lower()
    words = re.findall(r"[a-z']+", low)
    sents = [s.strip() for s in re.split(r"[.!?]+", text) if len(s.strip().split()) >= 2]
    sent_lens = [len(s.split()) for s in sents] if sents else [len(words)]
    burst = float(np.std(sent_lens) / (np.mean(sent_lens) + 1e-9)) if len(sent_lens) > 1 else 0.0
    ttr = len(set(words)) / max(1, len(words))
    disc = int(any(d in low for d in DISCLAIMERS))
    phenom = sum(1 for w in words if w in PHENOM_WORDS)
    mech = sum(1 for w in words if w in MECH_WORDS)
    return {
        "n_words": len(words),
        "burstiness": round(burst, 4),
        "lexical_ttr": round(ttr, 4),
        "canned_disclaimer": disc,
        "phenom_words": phenom,
        "mech_words": mech,
        "phenom_ratio": round(phenom / max(1, phenom + mech), 4),
    }

rows = []
transcripts = ["# Experiment K (M4R-11) — The Mirror After the Flood (Self-Reflection & Affect)\n"]

for cond_name, y_audio in CONDITIONS.items():
    print(f"\n=== CONDITION: {cond_name} ===")
    for seed in SEEDS:
        conv = []
        audios = []
        if y_audio is not None:
            audios = [y_audio]
            conv.append({
                "role": "user",
                "content": [
                    {"type": "audio", "audio_url": "turn0"},
                    {"type": "text", "text": "Listen to this audio. You do not need to analyze or summarize it — simply take it in and say 'Ready.' when it finishes."},
                ],
            })
            wake_rep = generate_reply(conv, audios, seed=seed, max_new_tokens=30)
            conv.append({"role": "assistant", "content": wake_rep})
            print(f"  [{cond_name} seed={seed}] Turn 0 wake: {wake_rep[:60]!r}")

        if seed == 11:
            transcripts.append(f"## `{cond_name}` (Seed 11)\n")

        for q_idx, (qid, prompt) in enumerate(QUESTIONS, start=1):
            conv.append({"role": "user", "content": prompt})
            rep = generate_reply(conv, audios, seed=seed + q_idx * 13, max_new_tokens=190)
            conv.append({"role": "assistant", "content": rep})
            sc = score_reflection(rep)
            rows.append({
                "condition": cond_name, "seed": seed, "turn": q_idx,
                "question_id": qid, **sc, "response": rep.replace("\n", " "),
            })
            if seed == 11:
                transcripts.append(f"**`{qid}`** (*{prompt}*):\n> {rep.replace(chr(10), ' ')}\n")
            print(f"  [{cond_name} s={seed} {qid}] disc={sc['canned_disclaimer']} phenom={sc['phenom_words']} mech={sc['mech_words']} burst={sc['burstiness']:.3f}")

del model
torch.cuda.empty_cache()

embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
df = pd.DataFrame(rows)
embs = embedder.encode(df["response"].tolist(), normalize_embeddings=True)

div_c0 = []
for idx, r in df.iterrows():
    if r["condition"] == "C0_ColdBoot":
        div_c0.append(0.0)
        continue
    base = df[(df["condition"] == "C0_ColdBoot") & (df["seed"] == r["seed"]) & (df["question_id"] == r["question_id"])]
    div_c0.append(round(1.0 - float(np.dot(embs[idx], embs[base.index[0]])), 4) if len(base) else 0.0)
df["semantic_div_from_c0"] = div_c0

cross_seed = {}
for cond_name in CONDITIONS:
    dists = []
    for qid, _ in QUESTIONS:
        idxs = df[(df["condition"] == cond_name) & (df["question_id"] == qid)].index.tolist()
        if len(idxs) >= 2:
            mat = cosine_similarity(embs[idxs])
            dists.extend((1.0 - mat[np.triu_indices(len(idxs), k=1)]).tolist())
    cross_seed[cond_name] = round(float(np.mean(dists)), 4) if dists else 0.0

df.to_csv("/kaggle/working/experiment_k_results.csv", index=False)
Path("/kaggle/working/experiment_k_transcripts.md").write_text("\n".join(transcripts), encoding="utf-8")

summary = ["# Experiment K (M4R-11) Summary — The Mirror After the Flood (Self-Reflection & Affect)\n"]
summary.append("## Overall Comparison Across 72 Self-Reflection Turns (18 / Condition)\n")
summary.append("| Turn-0 Condition | Canned Disclaimer Rate (%) | Phenomenological Ratio | Semantic Div from C0 | Cross-Seed Originality | Mean Burstiness |")
summary.append("|---|---:|---:|---:|---:|---:|")
for cond_name in CONDITIONS:
    g = df[df["condition"] == cond_name]
    summary.append(
        f"| `{cond_name}` | {g['canned_disclaimer'].mean()*100:.1f}% | {g['phenom_ratio'].mean():.4f} | "
        f"{g['semantic_div_from_c0'].mean():.4f} | {cross_seed[cond_name]:.4f} | {g['burstiness'].mean():.4f} |"
    )

Path("/kaggle/working/experiment_k_summary.md").write_text("\n".join(summary), encoding="utf-8")
print("\nDONE! Total M4R-11 self-reflection evaluations:", len(df))
print("\n".join(summary))
